# SteamCharts EDA Notebook
This notebook mirrors the CLI script but lets you iterate interactively for your report.

In [ ]:

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = "/mnt/data/steamcharts.csv"
FIG_DIR = Path("figs"); FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("out"); OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['month_dt'] = pd.to_datetime(df['month'], format='%b-%y', errors='coerce')
df.head(), df.shape


## 1. Basic Profile

In [ ]:
print('Shape:', df.shape)
df.dtypes

## 2. Missingness & Duplicates

In [ ]:
missing_pct = df.isna().mean().sort_values(ascending=False)*100
missing_pct.head(15)

In [ ]:
dup_count = df.duplicated().sum()
dup_count

## 3. Summary Statistics

In [ ]:

num_cols = df.select_dtypes(include=[np.number]).columns
num_desc = df[num_cols].describe().T
num_desc.to_csv(OUT_DIR/'numeric_summary.csv')
num_desc.head(10)


## 4. Visualizations

In [ ]:

df['avg_players'].dropna().hist(bins=50)
plt.title('Histogram: avg_players'); plt.xlabel('avg_players'); plt.ylabel('Count')
plt.tight_layout(); plt.savefig(FIG_DIR/'hist_avg_players.png'); plt.show()


In [ ]:

plt.boxplot(df['gain_percent'].dropna(), vert=True)
plt.title('Boxplot: gain_percent'); plt.ylabel('gain_percent')
plt.tight_layout(); plt.savefig(FIG_DIR/'box_gain_percent.png'); plt.show()


In [ ]:

sample = df.sample(min(5000, len(df)), random_state=42)
plt.scatter(sample['avg_players'], sample['peak_players'], s=5, alpha=0.5)
plt.title('avg_players vs peak_players (sample)')
plt.xlabel('avg_players'); plt.ylabel('peak_players')
plt.tight_layout(); plt.savefig(FIG_DIR/'scatter_avg_vs_peak.png'); plt.show()


In [ ]:

corr = df[num_cols].corr(numeric_only=True)
plt.imshow(corr, aspect='auto', interpolation='nearest'); plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Correlation Heatmap')
plt.tight_layout(); plt.savefig(FIG_DIR/'corr_heatmap.png'); plt.show()


In [ ]:

top3 = df.groupby('name')['avg_players'].mean().sort_values(ascending=False).head(3).index
ts = df[df['name'].isin(top3)].dropna(subset=['month_dt']).sort_values('month_dt')
for g in top3:
    sub = ts[ts['name']==g]
    plt.plot(sub['month_dt'], sub['avg_players'], label=g)
plt.title('Monthly avg_players — Top 3 Games')
plt.xlabel('Month'); plt.ylabel('avg_players'); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'timeseries_top3.png'); plt.show()


In [ ]:

top10 = df.groupby('name')['avg_players'].median().sort_values(ascending=False).head(10)
top10.sort_values().plot(kind='barh', figsize=(6,4))
plt.title('Top 10 by Median avg_players'); plt.xlabel('Median avg_players'); plt.ylabel('Game')
plt.tight_layout(); plt.savefig(FIG_DIR/'bar_top10_median_avg.png'); plt.show()


## 5. Save Key Artifacts
- Figures → `./figs`
- Tables  → `./out/numeric_summary.csv`